In [ ]:
from pathlib import Path
import csv


def merge_cfg_csvs_by_dataset(
    base_dir='/home/gabrimi8/RDL/ReDeLEx/logs/dbgnn_16combos_/',
    output_dir='/home/gabrimi8/RDL/ReDeLEx/plots',
):
    """
    For each dataset/task folder under base_dir:
    - find a model-run subfolder with cfg_*.csv files,
    - use header from the first CSV,
    - take only the second row (data row) from each CSV,
    - save one merged CSV into output_dir named <dataset>_<task>.csv.
    """
    base = Path(base_dir)
    out = Path(output_dir)

    if not base.exists():
        raise FileNotFoundError(f'Base directory not found: {base}')

    out.mkdir(parents=True, exist_ok=True)

    merged_files = []

    # Example dataset folder name: rel-amazon_item-churn
    dataset_folders = sorted([p for p in base.iterdir() if p.is_dir()])

    for dataset_folder in dataset_folders:
        # Prefer subfolders that contain cfg_*.csv, ignore run_logs.
        run_folders = sorted(
            [
                p
                for p in dataset_folder.iterdir()
                if p.is_dir() and p.name != 'run_logs' and any(p.glob('cfg_*.csv'))
            ]
        )

        if not run_folders:
            continue

        # Use the first matching run folder.
        run_folder = run_folders[0]

        def cfg_sort_key(path_obj):
            stem = path_obj.stem  # e.g., cfg_12
            try:
                return int(stem.split('_')[-1])
            except (ValueError, IndexError):
                return float('inf')

        csv_files = sorted(run_folder.glob('cfg_*.csv'), key=cfg_sort_key)
        if not csv_files:
            continue

        header = None
        rows = []

        for csv_file in csv_files:
            with open(csv_file, 'r', newline='') as f:
                reader = csv.reader(f)
                file_rows = list(reader)

            # Need at least header + one data row.
            if len(file_rows) < 2:
                print(f'Skipping {csv_file.name}: missing second row')
                continue

            if header is None:
                header = file_rows[0]

            rows.append(file_rows[1])

        if header is None or not rows:
            continue

        # Output name from dataset folder name split by first underscore.
        # rel-amazon_item-churn -> rel-amazon_item-churn.csv
        parts = dataset_folder.name.split('_', 1)
        if len(parts) == 2:
            out_name = f'{parts[0]}_{parts[1]}.csv'
        else:
            out_name = f'{dataset_folder.name}.csv'

        out_file = out / out_name
        with open(out_file, 'w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(header)
            writer.writerows(rows)

        merged_files.append(out_file)
        print(f'Wrote {out_file} ({len(rows)} rows)')

    print(f'Finished. Created {len(merged_files)} merged CSV file(s).')
    return merged_files


# Run once to generate merged CSVs for all dataset/task folders.
# merged_outputs = merge_cfg_csvs_by_dataset()
# merged_outputs

Wrote /home/gabrimi8/RDL/ReDeLEx/plots/ctu-adventureworks_adventureworks-original.csv (16 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/ctu-geneea_geneea-original.csv (4 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/ctu-lahman_lahman-original.csv (4 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_item-churn.csv (4 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_item-ltv.csv (4 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_user-churn.csv (4 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_user-ltv.csv (4 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-avito_ad-ctr.csv (8 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-avito_user-clicks.csv (8 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-avito_user-visits.csv (8 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-f1_driver-dnf.csv (16 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-f1_driver-position.csv (16 rows)
Wrote /home/gabrimi8/RDL/ReDeLEx/plots/rel-f1_driver-top3.csv (16 rows)
Wrote /home/gabrim

[PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/ctu-adventureworks_adventureworks-original.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/ctu-geneea_geneea-original.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/ctu-lahman_lahman-original.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_item-churn.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_item-ltv.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_user-churn.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-amazon_user-ltv.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-avito_ad-ctr.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-avito_user-clicks.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-avito_user-visits.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-f1_driver-dnf.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-f1_driver-position.csv'),
 PosixPath('/home/gabrimi8/RDL/ReDeLEx/plots/rel-f1_driver-top3.csv'),
 PosixPath('/home/gabr

In [ ]:
from pathlib import Path
import pandas as pd
import ipywidgets as widgets
from IPython.display import display


def load_filtered_plot_csv(csv_name_or_path, plots_dir='/home/gabrimi8/RDL/ReDeLEx/plots'):
    """
    Load one CSV from /plots and return a filtered DataFrame with:
    - all columns up to and including avg_training_time(_s)
    - best_test_roc_auc_mean + best_val_roc_auc_mean for classification
      OR
    - best_test_mae_mean + best_val_mae_mean for regression
    """
    path = Path(csv_name_or_path)
    if not path.is_absolute():
        path = Path(plots_dir) / path

    if not path.exists():
        raise FileNotFoundError(f'CSV not found: {path}')

    df = pd.read_csv(path)
    cols = list(df.columns)

    # Support both names just in case.
    cutoff_col = None
    for candidate in ('avg_training_time_s', 'avg_training_time'):
        if candidate in cols:
            cutoff_col = candidate
            break

    if cutoff_col is None:
        raise KeyError('Missing cutoff column: avg_training_time_s (or avg_training_time)')

    base_cols = cols[: cols.index(cutoff_col) + 1]

    classification_cols = ['best_test_roc_auc_mean', 'best_val_roc_auc_mean']
    regression_cols = ['best_test_mae_mean', 'best_val_mae_mean']

    has_classification = all(c in cols for c in classification_cols)
    has_regression = all(c in cols for c in regression_cols)

    if has_classification and not has_regression:
        metric_cols = classification_cols
    elif has_regression and not has_classification:
        metric_cols = regression_cols
    elif has_classification and has_regression:
        # If both sets exist, prefer classification columns.
        metric_cols = classification_cols
    else:
        raise KeyError(
            'Could not detect task type. Expected either ROC-AUC columns or MAE columns.'
        )

    selected_cols = base_cols + metric_cols
    return df[selected_cols].copy()


PLOTS_DIR = Path('/home/gabrimi8/RDL/ReDeLEx/plots')
csv_files = sorted([p.name for p in PLOTS_DIR.glob('*.csv')])

if not csv_files:
    raise FileNotFoundError(f'No CSV files found in: {PLOTS_DIR}')

file_dropdown = widgets.Dropdown(
    options=csv_files,
    value=csv_files[0],
    description='Dataset:',
    layout=widgets.Layout(width='500px'),
)
out = widgets.Output()


def on_dataset_change(change):
    if change['name'] == 'value' and change['new']:
        with out:
            out.clear_output(wait=True)
            df_table = load_filtered_plot_csv(change['new'], plots_dir=PLOTS_DIR)
            display(df_table)


file_dropdown.observe(on_dataset_change, names='value')

# Initial render
df_table = load_filtered_plot_csv(file_dropdown.value, plots_dir=PLOTS_DIR)
with out:
    display(df_table)

display(file_dropdown, out)


Dropdown(description='Dataset:', layout=Layout(width='500px'), options=('rel-amazon_item-churn.csv', 'rel-amaz…

Output()

In [16]:
CFG_MAP = {
    1:  dict(process_bridge=False, bridge_strategy='default',         process_hub=False, hub_strategy='default_combinations'),
    2:  dict(process_bridge=False, bridge_strategy='default',         process_hub=True,  hub_strategy='default_combinations'),
    3:  dict(process_bridge=False, bridge_strategy='default',         process_hub=True,  hub_strategy='keep_attributes'),
    4:  dict(process_bridge=False, bridge_strategy='default',         process_hub=True,  hub_strategy='keep_table'),
    5:  dict(process_bridge=True,  bridge_strategy='default',         process_hub=False, hub_strategy='default_combinations'),
    6:  dict(process_bridge=True,  bridge_strategy='keep_attributes', process_hub=False, hub_strategy='default_combinations'),
    7:  dict(process_bridge=True,  bridge_strategy='keep_table',      process_hub=False, hub_strategy='default_combinations'),
    8:  dict(process_bridge=True,  bridge_strategy='default',         process_hub=True,  hub_strategy='default_combinations'),
    9:  dict(process_bridge=True,  bridge_strategy='default',         process_hub=True,  hub_strategy='keep_attributes'),
    10: dict(process_bridge=True,  bridge_strategy='default',         process_hub=True,  hub_strategy='keep_table'),
    11: dict(process_bridge=True,  bridge_strategy='keep_attributes', process_hub=True,  hub_strategy='default_combinations'),
    12: dict(process_bridge=True,  bridge_strategy='keep_attributes', process_hub=True,  hub_strategy='keep_attributes'),
    13: dict(process_bridge=True,  bridge_strategy='keep_attributes', process_hub=True,  hub_strategy='keep_table'),
    14: dict(process_bridge=True,  bridge_strategy='keep_table',      process_hub=True,  hub_strategy='default_combinations'),
    15: dict(process_bridge=True,  bridge_strategy='keep_table',      process_hub=True,  hub_strategy='keep_attributes'),
    16: dict(process_bridge=True,  bridge_strategy='keep_table',      process_hub=True,  hub_strategy='keep_table'),
}

# Reverse lookup: (process_bridge, bridge_strategy, process_hub, hub_strategy) -> cfg number
_CFG_REVERSE = {
    (v['process_bridge'], v['bridge_strategy'], v['process_hub'], v['hub_strategy']): k
    for k, v in CFG_MAP.items()
}


def _to_bool(val):
    if isinstance(val, bool):
        return val
    return str(val).strip().lower() in ('true', '1', 'yes')


def _row_to_cfg_label(row):
    """
    Given a DataFrame row, read the run_params columns and look up the cfg number
    from _CFG_REVERSE. Falls back to 'cfg_?' if no match is found.
    """
    key = (
        _to_bool(row['process_bridge']),
        str(row['bridge_strategy']).strip(),
        _to_bool(row['process_hub']),
        str(row['hub_strategy']).strip(),
    )
    cfg_num = _CFG_REVERSE.get(key)
    if cfg_num is None:
        return f'cfg_? | PB:{key[0]} BS:{key[1]} PH:{key[2]} HS:{key[3]}'
    pb = 'T' if key[0] else 'F'
    ph = 'T' if key[2] else 'F'
    return f'cfg_{cfg_num}'


def build_summary_df(csv_name_or_path, plots_dir=PLOTS_DIR):
    """
    Build a 2-row summary DataFrame (Val / Test) with:
      Dataset | Task | Split | Benchmark | Best cfg | Best metric | Best relative % | Worst cfg | Worst metric | Worst relative %

    - Benchmark  : val/test metric of cfg_1 (matched by run_params, not row position)
    - Best cfg   : best among non-benchmark rows by test metric
    - Worst cfg  : worst among non-benchmark rows by test metric
    - Relative % : how much better than Benchmark (positive = better)
    """
    path = Path(csv_name_or_path)
    if not path.is_absolute():
        path = Path(plots_dir) / path
    df = pd.read_csv(path)

    # Parse dataset and task from filename
    stem = Path(csv_name_or_path).stem
    parts = stem.split('_', 1)
    dataset = parts[0]
    task = parts[1] if len(parts) == 2 else ''

    # Detect metric columns
    cols = list(df.columns)
    if 'best_test_roc_auc_mean' in cols:
        test_col = 'best_test_roc_auc_mean'
        val_col  = 'best_val_roc_auc_mean'
        higher_is_better = True
    else:
        test_col = 'best_test_mae_mean'
        val_col  = 'best_val_mae_mean'
        higher_is_better = False

    # Benchmark = row matching cfg_1 parameters
    cfg1 = CFG_MAP[1]
    benchmark_mask = (
        df['process_bridge'].apply(_to_bool) == cfg1['process_bridge']
    ) & (
        df['bridge_strategy'].str.strip() == cfg1['bridge_strategy']
    ) & (
        df['process_hub'].apply(_to_bool) == cfg1['process_hub']
    ) & (
        df['hub_strategy'].str.strip() == cfg1['hub_strategy']
    )

    if not benchmark_mask.any():
        raise ValueError('cfg_1 (benchmark) row not found in CSV')

    benchmark_row  = df[benchmark_mask].iloc[0]
    benchmark_test = benchmark_row[test_col]
    benchmark_val  = benchmark_row[val_col]

    # Non-benchmark rows used for both best and worst selection
    non_benchmark_df = df[~benchmark_mask]
    if non_benchmark_df.empty:
        raise ValueError('No non-benchmark rows available to determine best/worst cfg')

    # Best and worst among non-benchmark rows only
    if higher_is_better:
        best_pos  = int(non_benchmark_df[test_col].values.argmax())
        worst_pos = int(non_benchmark_df[test_col].values.argmin())
    else:
        best_pos  = int(non_benchmark_df[test_col].values.argmin())
        worst_pos = int(non_benchmark_df[test_col].values.argmax())

    best_row  = non_benchmark_df.iloc[best_pos]
    worst_row = non_benchmark_df.iloc[worst_pos]

    best_cfg_label  = _row_to_cfg_label(best_row)
    worst_cfg_label = _row_to_cfg_label(worst_row)

    best_test  = best_row[test_col]
    best_val   = best_row[val_col]
    worst_test = worst_row[test_col]
    worst_val  = worst_row[val_col]

    def relative(metric, benchmark):
        if benchmark == 0:
            return float('nan')
        if higher_is_better:
            return (metric - benchmark) / abs(benchmark) * 100
        else:
            return (benchmark - metric) / abs(benchmark) * 100

    rows = [
        {
            'Dataset':          dataset,
            'Task':             task,
            'Split':            'Val',
            'Benchmark':        round(benchmark_val,  4),
            'Best cfg':         best_cfg_label,
            'Best metric':      round(best_val,       4),
            'Best relative %':  round(relative(best_val,   benchmark_val), 2),
            'Worst cfg':        worst_cfg_label,
            'Worst metric':     round(worst_val,      4),
            'Worst relative %': round(relative(worst_val,  benchmark_val), 2),
        },
        {
            'Dataset':          dataset,
            'Task':             task,
            'Split':            'Test',
            'Benchmark':        round(benchmark_test, 4),
            'Best cfg':         best_cfg_label,
            'Best metric':      round(best_test,      4),
            'Best relative %':  round(relative(best_test,  benchmark_test), 2),
            'Worst cfg':        worst_cfg_label,
            'Worst metric':     round(worst_test,     4),
            'Worst relative %': round(relative(worst_test, benchmark_test), 2),
        },
    ]
    return pd.DataFrame(rows)


# --- Interactive widget ---
_PLOTS_DIR = Path('/home/gabrimi8/RDL/ReDeLEx/plots')
_csv_files = sorted([p.name for p in _PLOTS_DIR.glob('*.csv')])

if not _csv_files:
    raise FileNotFoundError(f'No CSV files found in: {_PLOTS_DIR}')

_summary_dropdown = widgets.Dropdown(
    options=_csv_files,
    value=_csv_files[0],
    description='Dataset:',
    layout=widgets.Layout(width='500px'),
)
_summary_out = widgets.Output()


def _on_summary_change(change):
    if change['name'] == 'value' and change['new']:
        with _summary_out:
            _summary_out.clear_output(wait=True)
            display(build_summary_df(change['new'], plots_dir=_PLOTS_DIR))


_summary_dropdown.observe(_on_summary_change, names='value')

with _summary_out:
    display(build_summary_df(_summary_dropdown.value, plots_dir=_PLOTS_DIR))

display(_summary_dropdown, _summary_out)

Dropdown(description='Dataset:', layout=Layout(width='500px'), options=('rel-amazon_item-churn.csv', 'rel-amaz…

Output()